In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
n = 500

data = pd.DataFrame({
    "latencyMs": np.random.uniform(20, 500, n),
    "packetLossPercent": np.random.uniform(0, 20, n),
    "loadPercent": np.random.uniform(10, 100, n),
})

# Simple rule to mark risk
data["risk"] = (
    (data["latencyMs"] > 200).astype(int) +
    (data["packetLossPercent"] > 8).astype(int) +
    (data["loadPercent"] > 75).astype(int)
) >= 2

data.to_csv("network_data.csv", index=False)
data.head()

,latencyMs,packetLossPercent,loadPercent,risk
0,199.779257,13.963234,26.661964,False
1,476.342867,10.721927,58.771085,True
2,371.357092,6.190552,88.565125,True
3,307.356072,16.275900,75.900240,True
4,94.888947,13.694623,82.590503,True


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import joblib

# Split features and target
X = data[["latencyMs", "packetLossPercent", "loadPercent"]]
y = data["risk"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluate and save
print("Accuracy:", model.score(X_test, y_test))
joblib.dump(model, "risk_model.pkl")

Accuracy: 0.91


['risk_model.pkl']

In [4]:
# Sample inputs from a live server
sample_input = pd.DataFrame([{
    "latencyMs": 250,
    "packetLossPercent": 10,
    "loadPercent": 80
}])

risk_prob = model.predict_proba(sample_input)[0][1]
risk_score = round(risk_prob * 100, 2)
level = "high" if risk_score > 70 else "medium" if risk_score > 40 else "low"

print(f"Risk Score: {risk_score}% | Risk Level: {level}")

Risk Score: 77.1% | Risk Level: high


In [5]:
def choose_best_server(servers_metrics):
    for s in servers_metrics:
        # Lower latency and lower load produce a better score
        s["score"] = (100 - s["riskScore"]) - (s["latencyMs"] * 0.1) - (s["loadPercent"] * 0.2)
    
    best = max(servers_metrics, key=lambda s: s["score"])
    return {
        "action": "reroute",
        "toServer": best["serverId"],
        "reason": f"Lowest risk ({best['riskScore']}%) aur latency ({best['latencyMs']}ms)"
    }

def double_check(server_id):
    return True

def run_agent(servers_metrics):
    decision = choose_best_server(servers_metrics)
    decision["verified"] = double_check(decision["toServer"])
    return decision

# Test with dummy telemetry data
test_data = [
    {"serverId": "server-1", "latencyMs": 250, "loadPercent": 80, "riskScore": 85},
    {"serverId": "server-2", "latencyMs": 90, "loadPercent": 40, "riskScore": 20}
]

run_agent(test_data)

{'action': 'reroute',
 'toServer': 'server-2',
 'reason': 'Lowest risk (20%) aur latency (90ms)',
 'verified': True}